# Users data preprocessing

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import os
from pytz import timezone
import matplotlib.pyplot as plt

### Load data

In [ ]:
businesses = pd.read_parquet(f"/content/drive/MyDrive/7180/restaurants.parquet")
checkins_exploded = pd.read_parquet(f"/content/drive/MyDrive/7180/checkin.parquet")
reviews = pd.read_parquet(f"/content/drive/MyDrive/7180/reviews.parquet")
# users = pd.read_parquet(f"/content/drive/MyDrive/7180/user.parquet")
# tips = pd.read_parquet(f"/content/drive/MyDrive/7180/tip.parquet")

In [ ]:
len(reviews), len(businesses)

In [ ]:
businesses.head()

In [ ]:
reviews.head()

### Checking if enough users have given tips, so that that time can be used for checkin

In [ ]:
tips.user_id.nunique()

In [ ]:
reviews.user_id.nunique()

 Tips only cover 15% of your review users, which is not enough

### Checkin hour distribution

In [ ]:
reviews['checkin_hour'] = reviews['checkin_ts'].dt.hour

print(f"Unique users: {reviews['user_id'].nunique()}")
print(f"Unique businesses: {reviews['business_id'].nunique()}")

reviews['checkin_hour'].plot.hist(bins=24)

### Review date hour distribuution

In [ ]:
reviews['date'].dt.hour.plot.hist(bins=24)

### States available in the dataset

In [ ]:
print(businesses['state'].value_counts())

### Build user preference data

In [ ]:

## User Preference categories from Yelp
cuisines = {
    'thai', 'japanese', 'chinese', 'mexican', 'italian', 'indian',
    'korean', 'vietnamese', 'french', 'greek', 'mediterranean',
    'american (traditional)', 'american (new)', 'southern', 'cajun/creole',
    'caribbean', 'latin american', 'ethiopian', 'turkish', 'moroccan',
    'persian/iranian', 'lebanese', 'pakistani', 'filipino', 'malaysian',
    'taiwanese', 'szechuan', 'hawaiian', 'cuban', 'brazilian',
    'asian fusion', 'tex-mex', 'soul food',
}

food_types = {
    'pizza', 'ramen', 'sandwiches', 'seafood', 'steakhouses',
    'sushi bars', 'noodles', 'barbeque', 'breakfast & brunch',
    'coffee & tea', 'bakeries', 'desserts', 'ice cream & frozen yogurt',
    'juice bars & smoothies', 'fast food', 'diners', 'buffets',
    'food trucks', 'comfort food', 'chicken wings', 'tacos',
    'hot dogs', 'salad', 'soup',
}

dietary = {
    'vegan', 'vegetarian', 'gluten-free', 'halal', 'kosher',
}

# Explode and filter to cuisines only
business_cats = businesses[['business_id', 'categories']].copy()
business_cats['categories'] = business_cats['categories']
business_cats = business_cats.explode('categories')
business_cats['categories'] = business_cats['categories'].str.strip().str.lower()

### User visits to a specific business category

In [ ]:
user_visits = reviews.merge(business_cats, on='business_id')

In [ ]:
user_visits.head()

### User preferences estimation

In [ ]:
reviews['user_id'].nunique()

### Utility functions

In [ ]:
def build_preferences(user_visits, categories, group_name):
    filtered = user_visits[user_visits['categories'].astype(str).isin(categories)]
    counts = filtered.pivot_table(
        index='user_id', columns='categories', aggfunc='size', fill_value=0
    )
    # Reindex to ensure all categories present
    counts = counts.reindex(columns=sorted(categories), fill_value=0)
    normalized = counts.div(counts.sum(axis=1), axis=0).fillna(0)
    return normalized

### Build Cusine and food type

In [ ]:
cuisine_prefs = build_preferences(user_visits, cuisines, 'cuisine')
food_type_prefs = build_preferences(user_visits, food_types, 'food_type')
dietary_prefs = build_preferences(user_visits, dietary, 'dietary')

all_users = reviews['user_id'].unique()

cuisine_prefs = cuisine_prefs.reindex(all_users, fill_value=0)
food_type_prefs = food_type_prefs.reindex(all_users, fill_value=0)
dietary_prefs = dietary_prefs.reindex(all_users, fill_value=0)

# dietary_prefs ignored becuase its sparse
user_preferences = pd.concat([cuisine_prefs, food_type_prefs], axis=1)
print(f"Shape: {user_preferences.shape}")

In [ ]:
cuisine_prefs.head()

In [ ]:
food_type_prefs.head()

In [ ]:
dietary_prefs.head()

In [ ]:
# Percentage of zeros per group
print(f"Cuisine sparsity:   {(cuisine_prefs == 0).mean().mean():.2%}")
print(f"Food type sparsity: {(food_type_prefs == 0).mean().mean():.2%}")
print(f"Dietary sparsity:   {(dietary_prefs == 0).mean().mean():.2%}")

In [ ]:
print(f"Pref sparsity:   {(user_preferences == 0).mean().mean():.2%}")

In [ ]:
user_preferences.head()

## user_preferences column mean

In [ ]:
col_density = (user_preferences > 0).mean()
print(col_density.sort_values())

In [ ]:
col_density = (user_preferences > 0).mean()
col_density.sort_values().plot.barh(figsize=(10, 14))
plt.axvline(x=0.1, color='red', linestyle='--', label='10% threshold')
plt.xlabel('User density')
plt.title('Category density across users')
plt.legend()
plt.tight_layout()
plt.show()

Cuisines with low user preference in Yelp: 'ethiopian', 'turkish', 'moroccan',
    'persian/iranian', 'lebanese', 'filipino', 'malaysian',
    'taiwanese', 'szechuan', 'hawaiian', 'brazilian',
  

### Removing cuisines low preference and rebuilding preference data

In [ ]:
cuisines = {
    'thai', 'japanese', 'chinese', 'mexican', 'italian', 'indian',
    'korean', 'vietnamese', 'french', 'greek', 'mediterranean',
    'american (traditional)', 'american (new)', 'southern', 'cajun/creole',
    'caribbean', 'latin american',
     'pakistani', 'cuban',
    'asian fusion', 'tex-mex', 'soul food',
}


food_types = {
    'pizza', 'ramen', 'sandwiches', 'seafood', 'steakhouses',
    'sushi bars', 'noodles', 'barbeque', 'breakfast & brunch',
    'coffee & tea', 'bakeries', 'desserts', 'ice cream & frozen yogurt',
    'juice bars & smoothies', 'fast food', 'diners', 'buffets',
    'food trucks', 'comfort food', 'chicken wings', 'tacos',
    'hot dogs', 'salad', 'soup',
}

cuisines_and_food_types = {'thai', 'japanese', 'chinese', 'mexican', 'italian', 'indian',
    'korean', 'vietnamese', 'french', 'greek', 'mediterranean',
    'american (traditional)', 'american (new)', 'southern', 'cajun/creole',
    'caribbean', 'latin american',
     'pakistani', 'cuban',
    'asian fusion', 'tex-mex', 'soul food','pizza', 'ramen', 'sandwiches', 'seafood', 'steakhouses',
    'sushi bars', 'noodles', 'barbeque', 'breakfast & brunch',
    'coffee & tea', 'bakeries', 'desserts', 'ice cream & frozen yogurt',
    'juice bars & smoothies', 'fast food', 'diners', 'buffets',
    'food trucks', 'comfort food', 'chicken wings', 'tacos',
    'hot dogs', 'salad', 'soup'}


dietary = {
    'vegan', 'vegetarian', 'gluten-free', 'halal', 'kosher',
}

# Explode and filter to cuisines only
business_cats = businesses[['business_id', 'categories']].copy()
business_cats['categories'] = business_cats['categories']
business_cats = business_cats.explode('categories')
business_cats['categories'] = business_cats['categories'].str.strip().str.lower()


def build_preferences(user_visits, categories, group_name):
    filtered = user_visits[user_visits['categories'].astype(str).isin(categories)]
    counts = filtered.pivot_table(
        index='user_id', columns='categories', aggfunc='size', fill_value=0
    )
    # Reindex to ensure all categories present
    counts = counts.reindex(columns=sorted(categories), fill_value=0)
    normalized = counts.div(counts.sum(axis=1), axis=0).fillna(0)
    return normalized

cuisine_prefs = build_preferences(user_visits, cuisines, 'cuisine')
food_type_prefs = build_preferences(user_visits, food_types, 'food_type')
dietary_prefs = build_preferences(user_visits, dietary, 'dietary')

all_users = reviews['user_id'].unique()

cuisine_prefs = cuisine_prefs.reindex(all_users, fill_value=0)
food_type_prefs = food_type_prefs.reindex(all_users, fill_value=0)
dietary_prefs = dietary_prefs.reindex(all_users, fill_value=0)

# Concatenate
user_preferences = pd.concat([cuisine_prefs, food_type_prefs], axis=1)
print(f"Shape: {user_preferences.shape}")

In [ ]:
print(f"Pref sparsity:   {(user_preferences == 0).mean().mean():.2%}")

In [ ]:
col_density = (user_preferences > 0).mean()

In [ ]:
col_density.sort_values().plot.barh(figsize=(10, 14))
plt.axvline(x=0.10, color='red', linestyle='--', label='10% threshold')
plt.xlabel('User density')
plt.title('Category density across users')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
user_preferences.head()

In [ ]:
user_preferences.reset_index(inplace=True)

In [ ]:
train_data = reviews.merge(user_preferences, on='user_id', right_index=True)

print(f"Shape: {train_data.shape}")

In [ ]:
train_data['label'] = (train_data['stars'] >= 4).astype(int)
print(f"Rows: {len(train_data)}")
print(f"Users: {train_data['user_id'].nunique()}")
print(f"Businesses: {train_data['business_id'].nunique()}")
print(train_data['label'].value_counts(normalize=True))

In [ ]:
train_data.head()

In [ ]:
train_data.columns

In [ ]:
train_data.drop(columns=['review_id', 'user_id', 'business_id', 'stars', 'useful', 'funny',
       'cool', 'text', 'date','day_of_week', 'is_weekend',  'checkin_ts'], inplace=True)

In [ ]:
train_data.head()

In [ ]:
len(train_data)

In [ ]:
train_data.to_parquet('/content/drive/MyDrive/7180/users_data.parquet')